# Ragged MoE fused-kernel speedup experiment

Run the cells in order. The notebook stores code and results in Google Drive,
checks that the assigned GPU can run the Triton Pallas backend, performs a small
smoke test, and then launches the configurable sweep.


## 1. Optional JAX memory setting

Run this **before importing JAX**. Disabling preallocation can help when Colab has
other GPU users, although it can increase fragmentation. Leave the value as
`false` for the first run; restart the runtime before changing it later.


In [5]:
import os

# Must be set before importing JAX.
os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")
print("XLA_PYTHON_CLIENT_PREALLOCATE =", os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"])


XLA_PYTHON_CLIENT_PREALLOCATE = false


## 2. Mount Google Drive and create project folders


In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

PROJECT_DIR = Path("/content/drive/MyDrive/Scaling-Exercise/ragged_moe_speedup_experiment")
RESULTS_ROOT = PROJECT_DIR / "results"
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)

print("Project:", PROJECT_DIR)
print("Results:", RESULTS_ROOT)

## 3. Get the experiment code from the repo

The sweep runs the committed `src/ragged_moe_speedup_experiment.py` from the
kernel repo — no manual upload. The script is JAX-version portable (it has a
compat layer for the post-0.8 Pallas memory-op API), so Colab's preinstalled
JAX works as-is.

In [ ]:
REPO_URL = "https://github.com/KrushnaBhanushali/pallas-moe-kernel.git"  # <-- edit me
REPO_DIR = Path("/content/pallas-moe-kernel")

import os
if not REPO_DIR.exists():
    !git clone $REPO_URL $REPO_DIR
else:
    !git -C $REPO_DIR pull

CODE_DIR = REPO_DIR / "src"
print("Code:", CODE_DIR)

## 4. Verify the assigned GPU and JAX backend

The script uses the Triton Pallas backend, which needs an Ampere-or-newer GPU.
A T4 has compute capability 7.5 and should not be used for this experiment.


In [8]:
import subprocess
import jax
import jaxlib

print("JAX:", jax.__version__)
print("jaxlib:", jaxlib.__version__)
print("Backend:", jax.default_backend())
print("Devices:", jax.devices())

if jax.default_backend() != "gpu":
    raise RuntimeError(
        "JAX is not using a GPU. Select Runtime > Change runtime type > GPU "
        "and restart the notebook."
    )

gpu_line = subprocess.check_output(
    [
        "nvidia-smi",
        "--query-gpu=name,compute_cap,memory.total",
        "--format=csv,noheader,nounits",
    ],
    text=True,
).strip().splitlines()[0]

name, compute_capability, memory_mib = [part.strip() for part in gpu_line.split(",")]
compute_major = int(compute_capability.split(".")[0])
GPU_MEMORY_GIB = float(memory_mib) / 1024.0

print("GPU:", name)
print("Compute capability:", compute_capability)
print(f"GPU memory: {GPU_MEMORY_GIB:.2f} GiB")

if compute_major < 8:
    raise RuntimeError(
        f"{name} has compute capability {compute_capability}. "
        "Reconnect until Colab assigns an L4, A100, H100, or newer GPU."
    )

# The estimator already includes a 1.35x margin. Use only part of total GPU
# memory to leave room for compilation and allocator overhead.
SAFE_ESTIMATED_MEMORY_GIB = max(1.0, GPU_MEMORY_GIB * 0.60)
print(f"Conservative sweep limit: {SAFE_ESTIMATED_MEMORY_GIB:.2f} GiB")


JAX: 0.7.2
jaxlib: 0.7.2
Backend: gpu
Devices: [CudaDevice(id=0)]
GPU: NVIDIA L4
Compute capability: 8.9
GPU memory: 22.49 GiB
Conservative sweep limit: 13.50 GiB


## 5. Import the experiment module


In [9]:
import sys
import importlib

if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

import ragged_moe_speedup_experiment as experiment
experiment = importlib.reload(experiment)

print("Loaded:", experiment.__file__)


Loaded: /content/drive/MyDrive/Scaling-Exercise/ragged_moe_speedup_experiment/code/ragged_moe_speedup_experiment.py


## 6. Run a small compiled smoke test

This verifies compilation, numerical correctness, CSV saving, and plotting. Its
shape is deliberately small and is not the final performance experiment.


In [11]:
SMOKE_OUTPUT_DIR = RESULTS_ROOT / "smoke_test"

smoke_config = experiment.SweepConfig(
    M_values=(1024,),
    D_values=(128,),
    F_values=(512,),
    E_values=(4,),
    BM_values=(16,),
    BF_values=(16,),
    group_patterns=("balanced",),
    mode="full_grid",
    warmup=2,
    iterations=5,
    seed=0,
    shuffle_cases=False,
    interpret=False,
    input_dtype="float32",
    max_estimated_problem_memory_gib=SAFE_ESTIMATED_MEMORY_GIB,
    resume=True,
    clear_jax_caches_between_problem_shapes=True,
    max_cases=1,
)

smoke_results = experiment.run_experiment(smoke_config, SMOKE_OUTPUT_DIR)
display(smoke_results)


RAGGED MoE FUSED-KERNEL SPEEDUP EXPERIMENT
Backend:              gpu
Problem shapes:       1
Tile pairs/shape:     1
Maximum fused cases:  1
Output directory:     /content/drive/MyDrive/Scaling-Exercise/ragged_moe_speedup_experiment/results/smoke_test

----------------------------------------------------------------------------------------
[1/1] M=1,024, D=128, F=512, E=4, routing=balanced, groups=[256, 256, 256, 256], estimated_memory=0.007 GiB
  Benchmarking reference once for this problem shape...
  BM= 16, BF= 16 | ref=   0.1650 ms | fused=   0.1532 ms | speedup=  1.077x | ideal=  2.333x | ideal-captured=  46.14% | L2=4.470e-06 | valid

Completed successful cases: 1
New cases this invocation:  1
Results:                    /content/drive/MyDrive/Scaling-Exercise/ragged_moe_speedup_experiment/results/smoke_test/results.csv
Plots:                      /content/drive/MyDrive/Scaling-Exercise/ragged_moe_speedup_experiment/results/smoke_test/plots
Failures/skips:             /content/dr

,case_id,timestamp,backend,input_dtype,M,D,F,E,BM,BF,...,speedup_gap_from_ideal,incremental_fusion_benefit_fraction,ref_tokens_per_second,fused_tokens_per_second,ref_effective_tflops,fused_effective_tflops,fused_actual_tflops,ref_minimum_traffic_gbps,fused_minimum_traffic_gbps,fused_no_cache_traffic_gbps
0,8ffa4a0b8fdba321,2026-07-09T18:11:28.102437+00:00,gpu,float32,1024,128,512,4,16,16,...,1.256738,0.057447,6.206662e+06,6.682067e+06,1.627039,1.751664,1.751664,44.489357,20.527309,225.800399


## 7. Define the main structured experiment

Begin with `max_cases=10`. After verifying the output, change it to `None` and
rerun using a new output directory or keep the same directory to resume.


In [12]:
MAIN_OUTPUT_DIR = RESULTS_ROOT / "structured_float32"

main_config = experiment.SweepConfig(
    M_values=(128, 256, 512, 1024, 4096, 16384, 65536, 131072),
    D_values=(64, 128, 256, 512, 1024),
    F_values=(256, 512, 1024, 2048, 4096),
    E_values=(1, 2, 4, 8, 16),
    BM_values=(8, 16, 32, 64, 128),
    BF_values=(16, 32, 64, 128),
    group_patterns=("balanced", "mild", "skewed", "one_dominant"),

    baseline_M=16384,
    baseline_D=128,
    baseline_F=512,
    baseline_E=4,
    mode="structured",

    warmup=3,
    iterations=10,
    seed=0,
    shuffle_cases=True,
    interpret=False,
    input_dtype="float32",

    l2_relative_error_tolerance=1e-4,
    normalized_max_error_tolerance=1e-3,
    max_estimated_problem_memory_gib=SAFE_ESTIMATED_MEMORY_GIB,

    peak_compute_tflops=None,
    peak_bandwidth_gbps=None,
    shared_memory_limit_bytes=None,

    resume=True,
    clear_jax_caches_between_problem_shapes=True,
    max_by_shape_plots=100,

    # First test: 10 successful cases. Change to None for the complete sweep.
    max_cases=10,
)

print(main_config)


SweepConfig(M_values=(128, 256, 512, 1024, 4096, 16384, 65536, 131072), D_values=(64, 128, 256, 512, 1024), F_values=(256, 512, 1024, 2048, 4096), E_values=(1, 2, 4, 8, 16), BM_values=(8, 16, 32, 64, 128), BF_values=(16, 32, 64, 128), group_patterns=('balanced', 'mild', 'skewed', 'one_dominant'), baseline_M=16384, baseline_D=128, baseline_F=512, baseline_E=4, mode='structured', warmup=3, iterations=10, seed=0, shuffle_cases=True, interpret=False, input_dtype='float32', l2_relative_error_tolerance=0.0001, normalized_max_error_tolerance=0.001, max_estimated_problem_memory_gib=13.496484375, peak_compute_tflops=None, peak_bandwidth_gbps=None, shared_memory_limit_bytes=None, resume=True, clear_jax_caches_between_problem_shapes=True, max_by_shape_plots=100, max_cases=10)


## 8. Run the experiment

The runner saves `results.csv` after every successful case and appends every
skip/error to `failures.csv`. Recoverable OOM or resource errors therefore do not
stop the remaining configurations.


In [13]:
main_results = experiment.run_experiment(main_config, MAIN_OUTPUT_DIR)
print("Successful rows:", len(main_results))
display(main_results.head())


RAGGED MoE FUSED-KERNEL SPEEDUP EXPERIMENT
Backend:              gpu
Problem shapes:       80
Tile pairs/shape:     20
Maximum fused cases:  10
Output directory:     /content/drive/MyDrive/Scaling-Exercise/ragged_moe_speedup_experiment/results/structured_float32

----------------------------------------------------------------------------------------
[1/80] M=16,384, D=128, F=512, E=1, routing=skewed, groups=[16384], estimated_memory=0.074 GiB
  Benchmarking reference once for this problem shape...
  BM= 32, BF= 64: FAILED XlaRuntimeError: RESOURCE_EXHAUSTED: Shared memory size limit exceeded: requested 155648, available: 101376
  BM= 32, BF= 16 | ref=   0.5729 ms | fused=   0.4106 ms | speedup=  1.395x | ideal=  4.879x | ideal-captured=  28.60% | L2=4.768e-06 | valid
  BM= 32, BF=128: FAILED XlaRuntimeError: RESOURCE_EXHAUSTED: Shared memory size limit exceeded: requested 294912, available: 101376
  BM= 16, BF=128: FAILED XlaRuntimeError: RESOURCE_EXHAUSTED: Shared memory size limit e

,case_id,timestamp,backend,input_dtype,M,D,F,E,BM,BF,...,speedup_gap_from_ideal,incremental_fusion_benefit_fraction,ref_tokens_per_second,fused_tokens_per_second,ref_effective_tflops,fused_effective_tflops,fused_actual_tflops,ref_minimum_traffic_gbps,fused_minimum_traffic_gbps,fused_no_cache_traffic_gbps
0,c8ceb0902f5c07c9,2026-07-09T18:19:51.623414+00:00,gpu,float32,16384,128,512,1,32,16,...,3.483435,0.101927,2.859976e+07,3.990676e+07,7.497255,10.461318,10.461318,147.345948,42.141539,694.696892
1,d6f6f0b0af8bd6fd,2026-07-09T18:19:52.354020+00:00,gpu,float32,16384,128,512,1,8,16,...,4.116601,-0.061311,2.859976e+07,2.179837e+07,7.497255,5.714311,5.714311,147.345948,23.019074,1450.899183
2,3f219868dd2f660f,2026-07-09T18:19:52.862281+00:00,gpu,float32,16384,128,512,1,8,32,...,4.076312,-0.050924,2.859976e+07,2.295060e+07,7.497255,6.016363,6.016363,147.345948,24.235838,1527.592190
3,477f32f00ad10726,2026-07-09T18:19:53.452890+00:00,gpu,float32,16384,128,512,1,16,16,...,3.653250,0.058146,2.859976e+07,3.505008e+07,7.497255,9.188167,9.188167,147.345948,37.012880,1184.412146
4,1f072bff990e06e2,2026-07-09T18:19:54.479497+00:00,gpu,float32,16384,128,512,1,128,16,...,3.464357,0.106845,2.859976e+07,4.045237e+07,7.497255,10.604347,10.604347,147.345948,42.717706,207.116151


## 9. Continue from the first 10 cases to the full sweep

Run this only after the first test succeeds. It keeps the same output directory,
so the first ten successful configurations are skipped automatically.


In [14]:
from dataclasses import replace

full_config = replace(main_config, max_cases=None)
main_results = experiment.run_experiment(full_config, MAIN_OUTPUT_DIR)
print("Total successful rows:", len(main_results))


Resuming with 10 completed fused cases.
RAGGED MoE FUSED-KERNEL SPEEDUP EXPERIMENT
Backend:              gpu
Problem shapes:       80
Tile pairs/shape:     20
Maximum fused cases:  1,600
Output directory:     /content/drive/MyDrive/Scaling-Exercise/ragged_moe_speedup_experiment/results/structured_float32

----------------------------------------------------------------------------------------
[1/80] M=16,384, D=128, F=512, E=1, routing=skewed, groups=[16384], estimated_memory=0.074 GiB
  Benchmarking reference once for this problem shape...
  BM=  8, BF=128: FAILED XlaRuntimeError: RESOURCE_EXHAUSTED: Shared memory size limit exceeded: requested 270336, available: 101376
  BM= 64, BF= 32: FAILED XlaRuntimeError: RESOURCE_EXHAUSTED: Shared memory size limit exceeded: requested 106496, available: 101376
  BM= 32, BF=128: FAILED XlaRuntimeError: RESOURCE_EXHAUSTED: Shared memory size limit exceeded: requested 294912, available: 101376
  BM= 16, BF= 64: FAILED XlaRuntimeError: RESOURCE_EXH

## 10. Inspect successful results and failures


In [ ]:
import pandas as pd

results_csv = MAIN_OUTPUT_DIR / "results.csv"
failures_csv = MAIN_OUTPUT_DIR / "failures.csv"

if results_csv.exists():
    results_df = pd.read_csv(results_csv)
    print("Successful cases:", len(results_df))
    display(
        results_df.sort_values("speedup_ref_over_fused", ascending=False)[
            [
                "M", "D", "F", "E", "BM", "BF", "group_pattern",
                "ref_median_ms", "fused_median_ms",
                "speedup_ref_over_fused", "ideal_memory_speedup",
                "speedup_percent_of_ideal", "l2_relative_error",
                "numerically_acceptable",
            ]
        ].head(30)
    )

if failures_csv.exists():
    failures_df = pd.read_csv(failures_csv)
    print("Failures/skips:", len(failures_df))
    display(failures_df.tail(30))
    if "error_category" in failures_df.columns:
        display(
            failures_df.groupby(
                ["stage", "error_category"],
                dropna=False,
            ).size().reset_index(name="count")
        )
else:
    print("No failures.csv was produced.")


## 11. Display generated figures


In [ ]:
from IPython.display import Image, display

plots_dir = MAIN_OUTPUT_DIR / "plots"
for plot_path in sorted(plots_dir.glob("*.png")):
    print(plot_path.name)
    display(Image(filename=str(plot_path)))


## 12. Zip the results for local download


In [ ]:
import shutil
from google.colab import files

zip_base = Path("/content") / MAIN_OUTPUT_DIR.name
zip_path = shutil.make_archive(
    str(zip_base),
    "zip",
    root_dir=str(MAIN_OUTPUT_DIR),
)
print("Created:", zip_path)
files.download(zip_path)
